# Demo 3: Vault Agent Injector (Sidecar Pattern) en Docker Desktop

En este notebook usaremos el **Vault Agent Injector** para inyectar secretos en un Pod de aplicación mediante un **sidecar**.

Requisitos previos:

- Haber ejecutado `1_vault_basico_docker_desktop.ipynb` (Vault instalado en modo dev).
- Tener el `VAULT_TOKEN` raíz y `VAULT_ADDR` exportados en tu entorno.
- (Opcional) Haber visto `2_kubernetes_auth_method.ipynb` para entender Kubernetes Auth.


In [ ]:
%%bash
set -euo pipefail

NAMESPACE=vault

echo "==> Habilitando el Vault Agent Injector via Helm (upgrade)"
cat > values-injector.yaml <<EOF
server:
  dataStorage:
    enabled: false
  dev:
    enabled: true
  service:
    type: ClusterIP
  readinessProbe:
    enabled: false
  livenessProbe:
    enabled: false
Injector:
  enabled: true
  externalVaultAddr: "http://vault.vault.svc.cluster.local:8200"
EOF

helm upgrade --install vault hashicorp/vault \
  --namespace "$NAMESPACE" \
  -f values-injector.yaml

echo "==> Comprobando pods en namespace vault"
kubectl -n "$NAMESPACE" get pods


In [ ]:
%%bash
set -euo pipefail

: "${VAULT_ADDR:?Debes exportar VAULT_ADDR}"
: "${VAULT_TOKEN:?Debes exportar VAULT_TOKEN}"

echo "==> Creando secreto de ejemplo en Vault"
vault kv put secret/webapp/config username="web-user" password="super-secret"

echo "==> Creando policy para que la app pueda leer el secreto"
vault policy write webapp-policy - <<EOF
path "secret/data/webapp/config" {
  capabilities = ["read"]
}
EOF

echo "==> Creando role de Kubernetes Auth para la app (usaremos app-sa del namespace app)"
vault write auth/kubernetes/role/webapp-role \
  bound_service_account_names=app-sa \
  bound_service_account_namespaces=app \
  policies=webapp-policy \
  ttl=1h


In [ ]:
%%bash
set -euo pipefail

NAMESPACE_APP=app

echo "==> Asegurando namespace app y ServiceAccount app-sa (si vienes del notebook 2, ya existen)"
kubectl create namespace "$NAMESPACE_APP" 2>/dev/null || echo "Namespace ya existe"
kubectl -n "$NAMESPACE_APP" create serviceaccount app-sa 2>/dev/null || echo "ServiceAccount ya existe"

echo "==> Creando Deployment ejemplo con anotaciones de Vault Agent Injector"
cat <<EOF | kubectl apply -f -
apiVersion: apps/v1
kind: Deployment
metadata:
  name: webapp-injected
  namespace: app
spec:
  replicas: 1
  selector:
    matchLabels:
      app: webapp-injected
  template:
    metadata:
      labels:
        app: webapp-injected
      annotations:
        vault.hashicorp.com/agent-inject: "true"
        vault.hashicorp.com/role: "webapp-role"
        vault.hashicorp.com/agent-inject-secret-config: "secret/data/webapp/config"
        vault.hashicorp.com/agent-inject-template-config: |
          {{- with secret "secret/data/webapp/config" -}}
          username={{ .Data.data.username }}
          password={{ .Data.data.password }}
          {{- end }}
    spec:
      serviceAccountName: app-sa
      containers:
      - name: app
        image: nginx:stable-alpine
        volumeMounts:
        - name: vault-secret
          mountPath: "/vault/secrets"
      volumes:
      - name: vault-secret
        emptyDir: {}
EOF

kubectl -n "$NAMESPACE_APP" get pods


## Verificar el secreto inyectado

Cuando el Pod esté en `Running`, ejecuta:

```bash
kubectl -n app get pods
POD_NAME=$(kubectl -n app get pod -l app=webapp-injected -o jsonpath='{.items[0].metadata.name}')
kubectl -n app exec -it "$POD_NAME" -- sh -c 'ls -R /vault && echo "---" && cat /vault/secrets/config || echo "aún no está"'
```

Deberías ver un archivo `/vault/secrets/config` con el contenido renderizado por la plantilla (usuario y contraseña) obtenido dinámicamente desde Vault mediante el Vault Agent sidecar.
